In [ ]:
import pandas as pd
import numpy as np
import os
import itertools as it
from snp_analysis_tools_sherlock import *
from coalescence_analysis_tools import *
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')

In [ ]:

def boot_gstats(gstats, bootstraps = 1000):
    meds = np.zeros(bootstraps)
    for b in range(bootstraps):
       # print(b)
        new_gstats= np.random.choice(gstats, size = len(gstats))
        meds[b] = np.median(new_gstats)
    return meds


In [ ]:
full_dfp7_gr = full_dfp7.groupby(['type_mesocosm','species_id','parent_subjects','media','inoculumn',]).median(numeric_only=True).reset_index()
full_dfp7_gr['winner_numeric'] = full_dfp7_gr['p7_s']>0.
full_dfp7_gr['counts']=1
full_dfp7_gr['het'] = full_dfp7_gr['winner_numeric']*(full_dfp7_gr['winner_numeric']-full_dfp7_gr['counts'])
#full_dfp7_gr_Good = full_dfp7_gr.loc[full_dfp7_gr['het']==0,:]
full_dfp7_gr_gr = full_dfp7_gr.groupby(['type_mesocosm']).sum(numeric_only=True).reset_index()
full_dfp7_gr_gr['fraction_one_winner'] = full_dfp7_gr_gr['winner_numeric']/full_dfp7_gr_gr['counts']
full_dfp7_gr_gr['Gstat'] =  get_Gstat(full_dfp7_gr_gr['winner_numeric'],full_dfp7_gr_gr['counts'])
full_dfp7_gr_gr.loc[full_dfp7_gr_gr['winner_numeric'] == 0,'Gstat'] = 2*full_dfp7_gr_gr['counts']*np.log(2)
full_dfp7_gr_gr.loc[full_dfp7_gr_gr['winner_numeric'] == full_dfp7_gr_gr['counts'],'Gstat'] = 2*full_dfp7_gr_gr['counts']*np.log(2)
full_dfp7_gr_gr['chi_cdf'] = chi2.cdf(full_dfp7_gr_gr['Gstat'], 1) 
full_dfp7_gr_gr['invchi_cdf'] = 1 - full_dfp7_gr_gr['chi_cdf'] 
#full_dfp7_gr_gr_Good = full_dfp7_gr_gr_Good.loc[full_dfp7_gr_gr_Good['counts']>4,:]
pvals = false_discovery_control(full_dfp7_gr_gr.sort_values(by='invchi_cdf')['invchi_cdf'].values)
#print(full_dfp7_gr_gr.sort_values(by='invchi_cdf')['type_mesocosm'])
#pvals
print(full_dfp7_gr_gr.sort_values(by='invchi_cdf')['invchi_cdf'].values)
print(pvals)
print(calculate_qvalues(full_dfp7_gr_gr.sort_values(by='invchi_cdf')['invchi_cdf'].values))
full_dfp7_gr['full_counts'] =  full_dfp7_gr['type_mesocosm'].transform(lambda x: \
                                            full_dfp7_gr_gr.loc[full_dfp7_gr_gr['type_mesocosm'] == x,'counts'].values[0])
full_dfp7_gr['relative_counts'] = full_dfp7_gr['counts']/full_dfp7_gr['full_counts'] 

full_dfp7_grplot = full_dfp7_gr.reset_index()
#full_dfp7_grplot = full_dfp7_grplot.loc[full_dfp7_grplot['full_counts']>4,:]
full_dfp7_grplot1 = full_dfp7_grplot.loc[full_dfp7_grplot['winner_numeric'] == 1,:].copy()
full_dfp7_grplot1['parent'] =full_dfp7_grplot1['parent_subjects'].transform(lambda x: x.split('-')[0]) 
full_dfp7_grplot2 = full_dfp7_grplot.loc[full_dfp7_grplot['winner_numeric'] == 0,:].copy()
full_dfp7_grplot2['parent'] =full_dfp7_grplot2['parent_subjects'].transform(lambda x: x.split('-')[1]) 
not_sig_mesos= full_dfp7_gr_gr.sort_values(by='invchi_cdf')['type_mesocosm'].values[pvals>.05]

full_dfp7_gr_gr_not_sig = full_dfp7_gr_gr.loc[full_dfp7_gr_gr['type_mesocosm'].isin(not_sig_mesos),:]
#print(len(not_sig_mesos))
#print(len(full_dfp7_gr_gr))
new_gstats = boot_gstats(full_dfp7_gr_gr_not_sig['Gstat'].values)
print(np.percentile(new_gstats,97.5))
print(np.percentile(new_gstats,2.5))
print(np.median(full_dfp7_gr_gr_not_sig['Gstat'].values), np.percentile(full_dfp7_gr_gr_not_sig['Gstat'].values, 90))
full_dfp7_grplotfull = pd.concat([full_dfp7_grplot1, full_dfp7_grplot2]).sort_values(by='type_mesocosm',ascending=False)


bars = hv.Bars(full_dfp7_grplotfull , kdims=['type_mesocosm','parent'],
               vdims = ['relative_counts'])

bars.opts(width=600,height = 400, ).opts(stacked=True, ylabel='Fraction winner', #xlabel='
                                         invert_axes=True,#xrotation = 90,
                                         cmap = bokeh.palettes.Set3[12])

bars.opts(legend_position='left')#4*6

In [ ]:

def boot_gstats(gstats, bootstraps = 1000):
    meds = np.zeros(bootstraps)
    for b in range(bootstraps):
       # print(b)
        new_gstats= np.random.choice(gstats, size = len(gstats))
        meds[b] = np.median(new_gstats)
    return meds


In [ ]:
full_dfp7_gr = full_dfp7.groupby(['type_mesocosm','species_id','parent_subjects','media','inoculumn',]).median(numeric_only=True).reset_index()
full_dfp7_gr['winner_numeric'] = full_dfp7_gr['p7_s']>0.
full_dfp7_gr['counts']=1
full_dfp7_gr['het'] = full_dfp7_gr['winner_numeric']*(full_dfp7_gr['winner_numeric']-full_dfp7_gr['counts'])
#full_dfp7_gr_Good = full_dfp7_gr.loc[full_dfp7_gr['het']==0,:]
full_dfp7_gr_gr = full_dfp7_gr.groupby(['type_mesocosm']).sum(numeric_only=True).reset_index()
full_dfp7_gr_gr['fraction_one_winner'] = full_dfp7_gr_gr['winner_numeric']/full_dfp7_gr_gr['counts']
full_dfp7_gr_gr['Gstat'] =  get_Gstat(full_dfp7_gr_gr['winner_numeric'],full_dfp7_gr_gr['counts'])
full_dfp7_gr_gr.loc[full_dfp7_gr_gr['winner_numeric'] == 0,'Gstat'] = 2*full_dfp7_gr_gr['counts']*np.log(2)
full_dfp7_gr_gr.loc[full_dfp7_gr_gr['winner_numeric'] == full_dfp7_gr_gr['counts'],'Gstat'] = 2*full_dfp7_gr_gr['counts']*np.log(2)
full_dfp7_gr_gr['chi_cdf'] = chi2.cdf(full_dfp7_gr_gr['Gstat'], 1) 
full_dfp7_gr_gr['invchi_cdf'] = 1 - full_dfp7_gr_gr['chi_cdf'] 
#full_dfp7_gr_gr_Good = full_dfp7_gr_gr_Good.loc[full_dfp7_gr_gr_Good['counts']>4,:]
pvals = false_discovery_control(full_dfp7_gr_gr.sort_values(by='invchi_cdf')['invchi_cdf'].values)
#print(full_dfp7_gr_gr.sort_values(by='invchi_cdf')['type_mesocosm'])
#pvals
print(full_dfp7_gr_gr.sort_values(by='invchi_cdf')['invchi_cdf'].values)
print(pvals)
print(calculate_qvalues(full_dfp7_gr_gr.sort_values(by='invchi_cdf')['invchi_cdf'].values))
full_dfp7_gr['full_counts'] =  full_dfp7_gr['type_mesocosm'].transform(lambda x: \
                                            full_dfp7_gr_gr.loc[full_dfp7_gr_gr['type_mesocosm'] == x,'counts'].values[0])
full_dfp7_gr['relative_counts'] = full_dfp7_gr['counts']/full_dfp7_gr['full_counts'] 

full_dfp7_grplot = full_dfp7_gr.reset_index()
#full_dfp7_grplot = full_dfp7_grplot.loc[full_dfp7_grplot['full_counts']>4,:]
full_dfp7_grplot1 = full_dfp7_grplot.loc[full_dfp7_grplot['winner_numeric'] == 1,:].copy()
full_dfp7_grplot1['parent'] =full_dfp7_grplot1['parent_subjects'].transform(lambda x: x.split('-')[0]) 
full_dfp7_grplot2 = full_dfp7_grplot.loc[full_dfp7_grplot['winner_numeric'] == 0,:].copy()
full_dfp7_grplot2['parent'] =full_dfp7_grplot2['parent_subjects'].transform(lambda x: x.split('-')[1]) 
not_sig_mesos= full_dfp7_gr_gr.sort_values(by='invchi_cdf')['type_mesocosm'].values[pvals>.05]

full_dfp7_gr_gr_not_sig = full_dfp7_gr_gr.loc[full_dfp7_gr_gr['type_mesocosm'].isin(not_sig_mesos),:]
#print(len(not_sig_mesos))
#print(len(full_dfp7_gr_gr))
new_gstats = boot_gstats(full_dfp7_gr_gr_not_sig['Gstat'].values)
print(np.percentile(new_gstats,97.5))
print(np.percentile(new_gstats,2.5))
print(np.median(full_dfp7_gr_gr_not_sig['Gstat'].values), np.percentile(full_dfp7_gr_gr_not_sig['Gstat'].values, 90))
full_dfp7_grplotfull = pd.concat([full_dfp7_grplot1, full_dfp7_grplot2]).sort_values(by='type_mesocosm',ascending=False)


bars = hv.Bars(full_dfp7_grplotfull , kdims=['type_mesocosm','parent'],
               vdims = ['relative_counts'])

bars.opts(width=600,height = 400, ).opts(stacked=True, ylabel='Fraction winner', #xlabel='
                                         invert_axes=True,#xrotation = 90,
                                         cmap = bokeh.palettes.Set3[12])

bars.opts(legend_position='left')#4*6

In [ ]:
fname = '~/git/coalescence-pilot-mgx/workflow/out/midas2_output/old_species/species/metadata.tsv'
df_metadata= pd.read_csv(fname, delimiter = '\t')
df_metadata

def transform_df(df_abundance):
    df_abundance['Lineage'] = df_abundance['species_id'].transform(lambda x: df_metadata.loc[df_metadata['species_id'] == x,'Lineage'].values[0])
    df_abundance['species'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-1])
    df_abundance['genus'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-2])
    df_abundance['family'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-3])
    df_abundance['phyla'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[1])
    return df_abundance

df_metadata  =transform_df(df_metadata)

df_abundance = pd.read_csv('e003_coalescence_metadata_round4_abundances.csv').drop(columns='Unnamed: 0')
#df_abundance  =transform_df(df_abundance)

df_abundance['passage'].unique()

In [ ]:
def adjust_df(meso_df):
    for sample in meso_df['sample'].unique():
        full= meso_df.loc[meso_df['sample']==sample,'relative_abundance'].sum()
        print(sample,full)
        meso_df.loc[meso_df['sample']==sample,'relative_abundance']=  meso_df.loc[meso_df['sample']==sample,'relative_abundance']/full
        print(full,meso_df.loc[meso_df['sample']==sample,'relative_abundance'].sum())
    return meso_df
   

In [ ]:
cmap_family= {'f__Bacteroidaceae': '#8dd3c7',
 'f__Enterobacteriaceae': '#ffffb3',
 'f__Porphyromonadaceae': '#bebada',
 'f__Peptoniphilaceae': '#fb8072',
 'f__Oscillospiraceae': '#80b1d3',
 'f__Enterococcaceae': '#fdb462',
 'f__Tannerellaceae': '#b3de69',
 'f__Lachnospiraceae': '#fccde5',
 'f__Veillonellaceae': '#bc80bd',
 'f__Peptostreptococcaceae': '#ccebc5',
 'f__Acidaminococcaceae': '#ffed6f',
 'f__other': '#d9d9d9'}

In [ ]:
df_abundance.loc[df_abundance['parent_subjects']=='AA-AF','mesocosm'].unique()

In [ ]:
meso_to_look_at = 'E11-AA-AF-mGAM-mGAM'

df_meso=df_abundance.loc[df_abundance['mesocosm']==meso_to_look_at,:]
in_sample = df_abundance.loc[df_abundance['sample']==df_meso['inoculumn_sample'].values[0],'sample'].values[0]
if in_sample not in df_meso['sample'].unique():
    in_sample = df_abundance.loc[df_abundance['sample']==df_meso['inoculumn_sample'].values[0],:]
    df_meso = pd.concat([df_meso,in_sample])
df_meso['family_sp']=df_meso['family']+ '-' + df_meso['species']
df_meso['passage'].unique()
#df_meso['species_id']=df_meso['species_id'].astype(str)
df_meso['passage_plot']=df_meso['passage'].astype(str)
#df_meso=df_meso.sort_values(by='passage_plot')
df_meso['is_important']=0.1
df_meso.loc[df_meso['species_id']==101346,'is_important']=1.
#df_meso=df_meso.loc[df_meso['sample']!='D_C4_C5_AF_AE_mBHI_mBHI_6_S316',:]

good_families = list(cmap_family.keys())
good_families

df_meso.loc[df_meso['relative_abundance']<1e-2,'relative_abundance']=0
#print(df_meso['family'])
df_meso = adjust_df(df_meso)
#print(df_meso['family'])
df_meso['family_plot']=df_meso['family'].copy()

df_meso.loc[~df_meso['family'].isin(good_families),'family_plot']='f__other'
df_meso=df_meso.sort_values(by='family_plot')
#df_mesog=df_meso#.loc[df_meso['is_important']==1.,:]
df_meso_small =df_meso.loc[df_meso['passage'].isin([0,1,2,3,4,5,6,7]),:].sort_values(by='family_plot')
bars = hv.Bars(df_meso_small, kdims=[hv.Dimension('passage', values=[0,1,2,3,4,5,6,7]), 'species_id',],
               vdims = ['relative_abundance','family_plot',])

bars=bars.opts(width=1500, height=250).opts(stacked=True,#alpha='relative_abundance',
                                      color='family_plot',
                                      cmap=cmap_family,
                                      alpha=1.,
                                           line_color = 'grey',
                                           bar_width=2.8,
                                      xlabel='Passage',
                                      ylabel='Relative Abundance',
                                        show_legend=True,legend_position='bottom',
                                         )#.sort(by='passage_plot')

bars

In [ ]:
sp_good = df_meso_small['species_id'].unique()
sp_good

In [ ]:
#df_meso=df_meso.sort_values(by='family_plot')
#df_meso_small =df_meso.loc[df_meso['passage'].isin([0,1,2,3,4,5]),:]
sp_good = df_meso_small['species_id'].unique()
df_meso_small_adjust = df_meso_small.copy()
new_df = []
for passage in df_meso_small['passage'].unique():
    df_meso_smallg = df_meso_small.loc[df_meso_small['passage']==passage,:].copy()
    df_meso_smallg['passage']=passage+.2
    new_df.append(df_meso_smallg)
    df_meso_smallb = df_meso_small.loc[df_meso_small['passage']==passage,:].copy()
    df_meso_smallb['passage']=passage-.2
    new_df.append(df_meso_smallb)
new_df = pd.concat(new_df)
df_meso_small_big = pd.concat([new_df, df_meso_small])
overlay = []
for sp in sp_good:
    alpha = .1
    #if sp == 102506:
     #   alpha = 1.
    
    area = hv.Area(df_meso_small_big.loc[df_meso_small_big['species_id']==sp,:].sort_values(by='passage'), 
                   kdims=[hv.Dimension('passage',values=[0,1,2,3,4,5,6,7],)],
                    vdims=['relative_abundance','family_plot']).opts(alpha=alpha,line_color='grey',line_width=.1,
                     color=cmap_family[df_meso_small.loc[df_meso_small['species_id']==sp,'family_plot'].values[0]]) 
    overlay.append(area)


overlay = hv.Overlay(overlay)



p=hv.Area.stack(overlay)
p

In [ ]:
pp = hv.render((p*bars).opts(xlim=(-0.25,7.5),ylim=(0,1),show_legend=True, height=400,width=600))
pp.output_backend = "svg"
bokeh.io.export_svg(pp, filename="plot.svg")
import svglib.svglib as svglib
from reportlab.graphics import renderPDF

#!rsvg-convert -f pdf -o file2.pdf plot.svg

In [ ]:
os.system("rsvg-convert -f pdf -o file23.pdf plot.svg")